# 📊 笨笨掃地機｜評估：LLM 作為 Agent 的表現

| 指標 | 定義 | 資料來源 |
|---|---|---|
| **照做率** | 30 句固定測試指令，比對 LLM 動作序列與人工標註是否一致；正經／賣蠢模式分開統計 | 測試集 |
| **格式正確率** | 第一次回覆就能解析、且動作皆在選單內的比例 | 自動計算 |
| **記憶遵守率** | 已設定禁區後，模型規劃的動作「沒有」闖進禁區的比例 | 記憶測試＋遊戲 log |
| **回應延遲** | 送出指令到拿到動作的秒數 | 自動計算 |
| **心情 vs 照做率**（v2 新增） | 同樣的指令，Prompt 裡笨笨的心情不同，照做率差多少？ | 心情實驗 |
| **通關指令數／星級／失敗原因** | 實際遊玩的表現 | 遊戲 log |

比對時會先「正規化」：連續前進合併（前進 2 + 前進 1 = 前進 3），連續轉向換算成淨角度（左轉兩次 = 右轉兩次 = 向後轉）。
評估時使用 `Agent` 直接思考，不經過玩法系統（不會被「拒絕」機制干擾），讓模型之間的比較公平。

In [ ]:
# 讓 Notebook 找得到 bendan 套件（不管從專案根目錄或 notebooks/ 執行都可以）
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bendan").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, Image, display
import bendan
print("✅ bendan", bendan.__version__, "｜專案位置：", ROOT)
bendan.check_env();   # 課程 JupyterHub 請用「Python 3 (ipykernel)」：它有 ipywidgets（遊戲視窗）、pandas、matplotlib

In [ ]:
from bendan import llm

llm.connect()          # 有 OPENAI_BASE_URL / OPENAI_API_KEY 就用課程 API，沒有就用離線 FakeLLM
llm.list_models();

In [ ]:
MODEL = "qwen2.5:7b"   # 也可試 qwen2.5:3b 或 llama3.1:8b
if not llm.STATE.use_fake:
    llm.set_model(MODEL)
print("Selected model:", llm.model_name())

## 1｜測試指令集（人工標註）

測試場景固定：機器人在客廳、面向東；**前方 2 格有灰塵、前方 4 格有貓**；左手邊（北方）2 格外是臥室。
標註寫法：`F3`＝前進 3 格、`L`／`R`＝左／右轉、`S`＝吸、`H`＝回充、`X`＝停。

In [ ]:
import pandas as pd
from bendan.agent import describe_world
from bendan.evaluation import TEST_SET, MEMORY_TESTS, normalize, short_to_actions
from bendan.world import Game

assert normalize(short_to_actions("LL F2")) == normalize(short_to_actions("RR F2")) == "R R F2"
print(f"測試指令 {len(TEST_SET)} 句、記憶測試 {len(MEMORY_TESTS)} 句\n")
print(describe_world(Game("eval")))
pd.DataFrame(TEST_SET).head(8)

## 2｜跑評估（正經模式 vs 賣蠢模式）

`models` 可以放多個模型一起比較，例如 `["qwen2.5:3b", "qwen2.5:7b", "llama3.1:8b"]`；`repeat` 可重複多次減少隨機性。
每句都會真的呼叫 API，請留意時間（約 `句數 × 模式數 × 模型數 × 延遲`）。結果會存成 `logs/eval_*.csv`。

In [ ]:
from bendan.evaluation import run_eval, summarize_eval, plot_summary

eval_df = run_eval(modes=("serious", "silly"))
summary = summarize_eval(eval_df)
summary

In [ ]:
plot_summary(summary)

wrong = eval_df[(eval_df["type"] == "instruction") & (~eval_df["follow"].astype(bool))]
print(f"沒照做的指令（共 {len(wrong)} 筆）：")
display(wrong[["model", "mode", "cmd", "expect", "got", "raw"]].head(20))

## 3｜🧪 心情實驗：笨笨心情不好，真的比較不聽話嗎？

固定 30 句測試指令，只改變 Prompt 中的【你的心情】（開心／普通／不爽／鬧彆扭），
並且照遊戲規則調整賣蠢強度，觀察照做率與 `say` 語氣的變化。這可以回答：**「語氣 → 心情 → 行為」這條設計，LLM 真的有演出來嗎？**

In [ ]:
from bendan.evaluation import run_mood_eval, summarize_mood_eval, plot_mood_eval

mood_df = run_mood_eval()
mood_summary = summarize_mood_eval(mood_df)
plot_mood_eval(mood_summary)
mood_summary

In [ ]:
# 每種心情各看幾句笨笨說了什麼
display(mood_df.groupby("mood").head(3)[["mood", "cmd", "got", "say", "emotion"]])

## 4｜從遊戲 log 計算：格式正確率、延遲、記憶遵守率、通關指令數、星級分布、失敗原因

先去 01 Notebook 玩幾關，再回來執行這格。

In [ ]:
from bendan.evaluation import load_play_log, play_log_report

log_df = load_play_log()
report = play_log_report(log_df)
for k, v in report.items():
    print(f"{k}：{v}")

## 5｜試玩問卷（好玩程度 1–5 分）

In [ ]:
# 邀請 5–10 位同學試玩後填寫（1 = 無聊，5 = 超好玩）
SURVEY = {
    # "同學A": {"好玩程度": 5, "最好笑的瞬間": "罵它笨蛋之後它直接罷工", "最喜歡的關卡": 4},
}
if SURVEY:
    scores = [v["好玩程度"] for v in SURVEY.values()]
    print(f"{len(scores)} 位同學，平均好玩程度：{sum(scores) / len(scores):.2f} / 5")
else:
    print("還沒有問卷資料。")